[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/05_scaling_laws/05_scaling_laws.ipynb)

# 05 · Scaling Laws：亲手拟合

配套讲解：`05_讲解.html` ｜ 课程：LLM 内核 ｜ <span style="color:#888">MODULE 05 / 8</span>

**本 notebook 全程 CPU，总耗时约 5 分钟。** 你将复刻 scaling law 研究的完整工作流——不是看别人的图，而是自己造数据点、自己拟合、自己外推：

| 实验 | 做什么 | 对应论文 |
|---|---|---|
| 实验一 L(N) | 真训 4 个尺寸的 mini-GPT（n_embd=16/32/64/128），`curve_fit` 拟合 $L(N)=A/N^\alpha+E$ 并外推 | [Kaplan 2020] |
| 实验二 L(D) | 固定中等模型，改变可用数据量（1/8 → 全量），拟合 $L(D)=B/D^\beta+E$ | [Kaplan 2020] |
| FLOPs 记账 | 逐层数矩阵乘法，验证 $C\approx 6ND$ | — |
| Chinchilla 计算器 | 给定算力 $C$ 解 compute-optimal $N^*, D^*$；画 iso-FLOP 曲线 | [Hoffmann 2022] |
| ✏️ 3 道练习 | `flops` 记账 / `fit_power_law` / `chinchilla_optimal` 解析解 | — |

模型复用模块 03 的 mini-GPT（下方内嵌精简版）；语料用内嵌的模板生成器——它有**已知的不可约熵**（句子里的随机选择无法被预测），所以拟合出的 $E>0$ 有真实含义。

In [ ]:
import math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

torch.manual_seed(42); np.random.seed(42)

# ---- 内嵌小语料：模板句生成器（确定性种子）----
# 每句 = 主语×谓语×宾语×状语 的随机组合：模型能学会拼写/语法，
# 但每个选择点的随机性是不可约熵 E 的来源——正适合演示 L = A/N^alpha + E
SUBJ = ["the model", "a network", "the agent", "our system", "the baseline",
        "a transformer", "the encoder", "the critic"]
VERB = ["learns", "predicts", "encodes", "compresses", "generates",
        "evaluates", "samples", "forgets"]
OBJ  = ["the data", "long sequences", "rare tokens", "clean text",
        "the context", "next tokens", "hidden states", "the gradient"]
ADV  = ["quickly", "slowly", "reliably", "poorly", "smoothly",
        "eventually", "rarely", "everywhere"]

def make_corpus(n_chars, seed=20):
    rng = random.Random(seed)
    parts, total = [], 0
    while total < n_chars:
        s = rng.choice(SUBJ) + " " + rng.choice(VERB) + " " + \
            rng.choice(OBJ) + " " + rng.choice(ADV) + ". "
        parts.append(s); total += len(s)
    return "".join(parts)

corpus = make_corpus(120_000)
vocab = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)
n_split = int(0.9 * len(data))
train_ids, val_ids = data[:n_split], data[n_split:]

# 理论不可约熵下界：每句 4 个独立选择、各 8 选 1 → 12 bit/句 ≈ 0.21 nat/char
sent_len = len(corpus) / corpus.count(". ")
H_floor = 4 * math.log(8) / sent_len
print(f"语料 {len(corpus):,} chars | vocab {len(vocab)} | train {len(train_ids):,} / val {len(val_ids):,}")
print(f"样例: {corpus[:120]}")
print(f"理论熵下界 ≈ {H_floor:.3f} nat/char（拟合出的 E 应略高于它）")

In [ ]:
# ---- 模块 03 的 mini-GPT（精简内嵌版）----
class Block(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        self.h = n_head
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(d, dim=2)
        q = q.view(B, T, self.h, d // self.h).transpose(1, 2)
        k = k.view(B, T, self.h, d // self.h).transpose(1, 2)
        v = v.view(B, T, self.h, d // self.h).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(y.transpose(1, 2).reshape(B, T, d))
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d, n_layer, n_head, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Parameter(torch.zeros(1, block_size, d))
        self.blocks = nn.ModuleList(Block(d, n_head) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:, :T]
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def count_nonemb_params(model):
    # Kaplan/Hoffmann 的 N 不含 token/pos embedding（这里也排除 unembedding head）
    skip = {id(model.tok_emb.weight), id(model.pos_emb), id(model.head.weight)}
    return sum(p.numel() for p in model.parameters() if id(p) not in skip)

def get_batch(ids, block, batch):
    ix = torch.randint(len(ids) - block - 1, (batch,))
    x = torch.stack([ids[i:i + block] for i in ix])
    y = torch.stack([ids[i + 1:i + block + 1] for i in ix])
    return x, y

def train_model(n_embd, n_layer, n_head, steps=400, data_frac=1.0,
                lr=2e-3, batch=32, block=64):
    torch.manual_seed(1234)
    u = max(block + 2, int(len(train_ids) * data_frac))   # 可用的独特训练 token 数
    tr = train_ids[:u]
    model = MiniGPT(len(vocab), n_embd, n_layer, n_head, block)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    # Hoffmann 的教训：cosine schedule 长度必须 == 实际训练步数，否则跨 (N,D) 不可比
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    model.train()
    for _ in range(steps):
        xb, yb = get_batch(tr, block, batch)
        _, loss = model(xb, yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval()
    torch.manual_seed(999)                                # 固定 val batch，模型间可比
    with torch.no_grad():
        vl = [model(*get_batch(val_ids, block, batch))[1].item() for _ in range(40)]
    return {"N": count_nonemb_params(model), "val_loss": float(np.mean(vl)),
            "unique_tokens": u, "tokens_seen": steps * batch * block}

print("mini-GPT 就绪。n_embd=64/2层 的非嵌入参数量 =",
      count_nonemb_params(MiniGPT(len(vocab), 64, 2, 4, 64)))

In [ ]:
# ============ 实验一：参数维度 L(N) ============
# 4 个尺寸 × 相同步数（每个 ~20-40 秒，共约 2 分钟）。
# 注意：所有模型 tokens_seen 相同 → 我们扫的是 N 这一个轴（Kaplan 的 L(N) 设定近似）
sizes = [(16, 1, 2), (32, 1, 2), (64, 2, 4), (128, 2, 8)]   # (n_embd, n_layer, n_head)
runs_N = []
for d, L_, h in sizes:
    t0 = time.time()
    r = train_model(d, L_, h)
    runs_N.append(r)
    print(f"n_embd={d:>3} n_layer={L_} | N={r['N']:>8,} | val_loss={r['val_loss']:.4f} | {time.time()-t0:.0f}s")

# ---- 拟合 L(N) = A/N^alpha + E，并外推 ----
def L_of_N(N, A, alpha, E):
    return A / N**alpha + E

Ns = np.array([r["N"] for r in runs_N], dtype=float)
Ls = np.array([r["val_loss"] for r in runs_N])
(A_N, alpha_N, E_N), _ = curve_fit(L_of_N, Ns, Ls, p0=[5.0, 0.3, 0.5 * Ls.min()],
                                   bounds=([1e-9, 0.01, 0.0], [1e9, 2.0, Ls.min()]),
                                   maxfev=20000)
print(f"\n拟合: L(N) = {A_N:.3g}/N^{alpha_N:.3f} + {E_N:.3f}   (E 对比理论熵下界 {H_floor:.3f})")

N_grid = np.logspace(np.log10(Ns.min() / 2), np.log10(Ns.max() * 30), 200)
N_big = Ns.max() * 10
pred_big = L_of_N(N_big, A_N, alpha_N, E_N)
plt.figure(figsize=(6.5, 4.2))
plt.loglog(Ns, Ls, "o", ms=8, label="4 次真实训练")
plt.loglog(N_grid, L_of_N(N_grid, A_N, alpha_N, E_N), "-",
           label=f"拟合 A/N^{alpha_N:.2f}+{E_N:.2f}")
plt.axhline(E_N, ls=":", c="gray", label=f"不可约损失 E={E_N:.2f}")
plt.loglog([N_big], [pred_big], "r*", ms=14, label=f"外推 10x: L≈{pred_big:.2f}")
plt.axvspan(Ns.max(), N_grid.max(), color="orange", alpha=0.08)
plt.xlabel("非嵌入参数量 N"); plt.ylabel("val loss (nat/char)")
plt.title("L(N)：log-log 下幂律近似直线，大 N 端弯向 E")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
print("⚠️ 只有 4 个点、跨 ~1.5 个数量级——真实论文用几十个模型跨 6+ 个数量级。阴影区=外推区，可信度随距离衰减。")

In [ ]:
# ============ 实验二：数据维度 L(D) ============
# 固定中等模型 (n_embd=64, 2 层)，只改变可用独特数据量 U：1/8 → 全量（共约 2 分钟）。
# U 小时模型反复刷同一批数据 → 过拟合 → val loss 高，这就是 B/D^beta 项
fracs = [0.125, 0.25, 0.5, 1.0]
runs_D = []
for f in fracs:
    t0 = time.time()
    r = train_model(64, 2, 4, data_frac=f)
    runs_D.append(r)
    print(f"data_frac={f:>5} | U={r['unique_tokens']:>7,} tokens | val_loss={r['val_loss']:.4f} | {time.time()-t0:.0f}s")

def L_of_D(D, B, beta, E):
    return B / D**beta + E

Ds = np.array([r["unique_tokens"] for r in runs_D], dtype=float)
LsD = np.array([r["val_loss"] for r in runs_D])
(B_D, beta_D, E_D), _ = curve_fit(L_of_D, Ds, LsD, p0=[5.0, 0.3, 0.5 * LsD.min()],
                                  bounds=([1e-9, 0.01, 0.0], [1e9, 2.0, LsD.min()]),
                                  maxfev=20000)
print(f"\n拟合: L(D) = {B_D:.3g}/D^{beta_D:.3f} + {E_D:.3f}")

D_grid = np.logspace(np.log10(Ds.min() / 2), np.log10(Ds.max() * 30), 200)
plt.figure(figsize=(6.5, 4.2))
plt.loglog(Ds, LsD, "s", ms=8, c="tab:green", label="4 次真实训练")
plt.loglog(D_grid, L_of_D(D_grid, B_D, beta_D, E_D), "-", c="tab:green",
           label=f"拟合 B/D^{beta_D:.2f}+{E_D:.2f}")
plt.axhline(E_D, ls=":", c="gray")
plt.xlabel("独特训练 token 数 D"); plt.ylabel("val loss (nat/char)")
plt.title("L(D)：数据越多 val loss 越低，同样服从幂律+下限")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"现在我们有了自己的联合定律常数: alpha={alpha_N:.3f}, beta={beta_D:.3f}（Hoffmann: 0.34 / 0.28）")

## FLOPs 记账与 compute-optimal 配比

有了 $L(N)$ 和 $L(D)$ 两条曲线，下一个问题就是 Chinchilla 之问：**给定算力 $C\approx 6ND$，怎么分配 $N$ 和 $D$？**（推导见讲解 §3–4）

$$N^*(C) = G\left(\tfrac{C}{6}\right)^{\frac{\beta}{\alpha+\beta}},\qquad D^*(C) = G^{-1}\left(\tfrac{C}{6}\right)^{\frac{\alpha}{\alpha+\beta}},\qquad G=\left(\tfrac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}}$$

下面三个 cell 依次：① 逐层数矩阵乘法验证 $C\approx6ND$ 的记账；② Chinchilla 配比计算器——用 [Hoffmann 2022] 公布常数和我们自己拟合的常数各算一版；③ 合成 iso-FLOP 曲线，标出谷底连线（Hoffmann 的方法 2）。

In [ ]:
# ============ C ≈ 6ND 记账与验证 ============
def flops_per_token_train(N):
    # 前向 2N（每参数 1 乘 1 加）+ 反向 4N（dL/dx 与 dL/dW 各一次同规模矩阵乘）
    return 6 * N

def total_train_flops(N, D):
    return 6 * N * D

# 逐层手工记账：扫模型里每个 nn.Linear（矩阵乘 = 每 token 2*in*out FLOPs）
def forward_flops_manual(model, T):
    f = sum(2 * m.in_features * m.out_features for m in model.modules()
            if isinstance(m, nn.Linear))
    d = model.tok_emb.embedding_dim
    f += len(model.blocks) * 4 * T * d   # QK^T 与 attn·V：无参数、与上下文长 T 成正比
    return f

m = MiniGPT(len(vocab), 128, 2, 8, 64)
N = count_nonemb_params(m)
fwd_manual = forward_flops_manual(m, T=64)
print(f"n_embd=128 模型: 非嵌入 N = {N:,}")
print(f"每 token 前向 (逐层记账) = {fwd_manual:,} FLOPs")
print(f"每 token 前向 (近似 2N)  = {2*N:,} FLOPs   比值 = {fwd_manual/(2*N):.2f}")
print("→ 比值略 >1：记账里含 head 投影与 attention 项，2N 近似把它们忽略了（讲解 §3）\n")

# 给我们最大的一次真实训练算总账
r = runs_N[-1]
C_run = total_train_flops(r["N"], r["tokens_seen"])
print(f"实验一最大模型: N={r['N']:,}, tokens_seen={r['tokens_seen']:,} → C ≈ {C_run:.2e} FLOPs")
print(f"对照量级: 1B 参数 × 20B token → C = {total_train_flops(1e9, 2e10):.1e}（≈14 GPU天 @100TFLOP/s）")

In [ ]:
# ============ Chinchilla 配比计算器 ============
def chinchilla_alloc(C, A, alpha, B, beta):
    a_exp = beta / (alpha + beta)
    b_exp = alpha / (alpha + beta)
    G = (alpha * A / (beta * B)) ** (1.0 / (alpha + beta))
    N_opt = G * (C / 6.0) ** a_exp
    D_opt = (C / 6.0) ** b_exp / G
    return N_opt, D_opt

HOFF = dict(A=406.4, alpha=0.34, B=410.7, beta=0.28)   # [Hoffmann 2022] 表 3（E=1.69）

print("―― 版本 A：Hoffmann 公布常数 ――")
for C in [1e21, 5.76e23, 1e25]:          # 5.76e23 = Chinchilla/Gopher 的训练算力
    Nopt, Dopt = chinchilla_alloc(C, **HOFF)
    print(f"C={C:.2e} → N*={Nopt:.2e}  D*={Dopt:.2e}  D/N={Dopt/Nopt:>6.1f}")
print("注意 D/N 在 50~120 之间而不是 20！“每参数 20 token”来自 iso-FLOP 谷底法（方法 2），")
print("参数化拟合的公布常数给出更大的比值——讲解 §4 讨论的方法间张力，亲眼可见。\n")

print("―― 版本 B：我们自己拟合的玩具常数 ――")
A_toy, B_toy = float(A_N), float(B_D)
for C in [1e11, 1e12, 1e13]:             # 玩具实验的算力量级
    Nopt, Dopt = chinchilla_alloc(C, A_toy, float(alpha_N), B_toy, float(beta_D))
    print(f"C={C:.0e} → N*={Nopt:.2e}  D*={Dopt:.2e}  D/N={Dopt/Nopt:>8.1f}")
print("玩具常数来自 4+4 个点的拟合，数值仅示意——但解析解的机制和 frontier lab 用的一模一样。")

In [ ]:
# ============ iso-FLOP 曲线（合成演示，Hoffmann 方法 2）============
# 用联合定律 L(N,D)=E+A/N^a+B/D^b 合成：固定 C，沿约束 D=C/(6N) 扫 N
E_h, A_h, al_h, B_h, be_h = 1.69, 406.4, 0.34, 410.7, 0.28

def L_joint(N, D):
    return E_h + A_h / N**al_h + B_h / D**be_h

plt.figure(figsize=(7, 4.4))
minima = []
for C, c in zip([1e21, 1e22, 1e23], ["tab:blue", "tab:orange", "tab:red"]):
    N_axis = np.logspace(8, 12.5, 400)
    D_axis = C / (6 * N_axis)
    losses = L_joint(N_axis, D_axis)
    i = int(np.argmin(losses))
    minima.append((N_axis[i], losses[i]))
    plt.semilogx(N_axis, losses, c=c, label=f"C={C:.0e}")
    plt.plot(N_axis[i], losses[i], "v", c=c, ms=10)
mN, mL = zip(*minima)
plt.plot(mN, mL, "k--", lw=1, label="谷底连线 → N*(C)")
plt.ylim(1.8, 3.4); plt.xlabel("N（约束 D=C/6N）"); plt.ylabel("L(N, C/6N)")
plt.title("iso-FLOP：每条曲线=一个算力预算；谷底=该预算的最优分配")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

# 谷底应落在解析解上
for (Nm, _), C in zip(minima, [1e21, 1e22, 1e23]):
    Nopt, _ = chinchilla_alloc(C, A_h, al_h, B_h, be_h)
    print(f"C={C:.0e}: 数值谷底 N={Nm:.2e} vs 解析 N*={Nopt:.2e}")
print("左支：模型太小（容量瓶颈）；右支：模型太大、数据太少（欠训练，GPT-3 在这一支）")

## ✏️ 练习 1：FLOPs 记账函数

实现两个函数（量级直觉是 scaling 工作的日常工具，值得肌肉记忆）：

- `flops_per_token(N)`：训练时每个 token 的 FLOPs，$\approx 6N$（前向 $2N$ + 反向 $4N$）；
- `total_flops(N, D)`：训练总算力 $C \approx 6ND$。

**提示**：各一行。自测里有一个你应该背下来的锚点：$N=10^9,\ D=2\times10^{10} \Rightarrow C=1.2\times10^{20}$。

In [ ]:
def flops_per_token(N):
    # TODO: 返回训练时每 token 的近似 FLOPs（前向 2N + 反向 4N）
    raise NotImplementedError

def total_flops(N, D):
    # TODO: 返回训练总 FLOPs（用 flops_per_token）
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
assert abs(flops_per_token(1e9) - 6e9) / 6e9 < 1e-9
assert abs(total_flops(1e9, 2e10) - 1.2e20) / 1.2e20 < 1e-9      # 必背锚点
assert abs(total_flops(7e10, 1.4e12) - 5.88e23) / 5.88e23 < 1e-9 # ≈ Chinchilla 的真实算力
assert total_flops(0, 1e12) == 0                                  # 边界
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `fit_power_law`

把实验一里的拟合封装成可复用函数：`fit_power_law(Ns, losses)` 拟合 $L(N) = A/N^{\alpha} + E$，返回 `(A, alpha, E)`。

**提示**（合理的 `p0` 是 `curve_fit` 成败的关键，10–15 行）：
1. 取 `E0 = 0.9 * losses.min()`（E 一定低于观察到的最小 loss）；
2. 幂律在 log-log 下是直线：对 `np.log(Ns)` 和 `np.log(losses - E0)` 用 `np.polyfit(..., 1)` 拿到斜率/截距 → `alpha0 = -斜率`，`A0 = exp(截距)`；
3. 用 `curve_fit(..., p0=[A0, alpha0, E0], bounds=([0,0,0],[np.inf, 5, losses.min()]), maxfev=20000)`。

In [ ]:
def fit_power_law(Ns, losses):
    # 返回 (A, alpha, E)，使 losses ≈ A / Ns**alpha + E
    Ns = np.asarray(Ns, dtype=float)
    losses = np.asarray(losses, dtype=float)
    # TODO: 1) 构造 p0（见提示）  2) curve_fit  3) 返回三元组
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测：在合成幂律数据上恢复参数（容差 10%）----
true_A, true_alpha, true_E = 406.4, 0.34, 1.69          # Hoffmann 的 L(N) 项
Ns_test = np.logspace(6, 11, 10)
losses_test = true_A / Ns_test**true_alpha + true_E
A_f, alpha_f, E_f = fit_power_law(Ns_test, losses_test)
assert abs(alpha_f - true_alpha) / true_alpha < 0.10, f"alpha={alpha_f}"
assert abs(E_f - true_E) / true_E < 0.10, f"E={E_f}"
assert abs(A_f - true_A) / true_A < 0.10, f"A={A_f}"
# 外推检查：用拟合参数预测一个没见过的 N
pred = A_f / 1e12**alpha_f + E_f
truth = true_A / 1e12**true_alpha + true_E
assert abs(pred - truth) / truth < 0.02
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `chinchilla_optimal`

实现讲解 §4 推出的解析解：`chinchilla_optimal(C, A, alpha, B, beta)` 返回 `(N_opt, D_opt)`：

$$N^* = G\left(\tfrac{C}{6}\right)^{\frac{\beta}{\alpha+\beta}},\quad D^* = G^{-1}\left(\tfrac{C}{6}\right)^{\frac{\alpha}{\alpha+\beta}},\quad G=\left(\tfrac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}}$$

**提示**：5–8 行。自测会检查三件事：① 解满足约束 $6N^*D^*=C$；② 对称情形 $\alpha=\beta, A=B$ 时 $N^*=D^*=\sqrt{C/6}$；③ 它确实是最小值（扰动配比的 loss 更高）。

In [ ]:
def chinchilla_optimal(C, A, alpha, B, beta):
    # 返回 (N_opt, D_opt)
    # TODO: 按公式计算 G、两个指数，再算 N_opt 与 D_opt
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
# ① 约束：6 N* D* == C
N1, D1 = chinchilla_optimal(1e21, 406.4, 0.34, 410.7, 0.28)
assert abs(6 * N1 * D1 - 1e21) / 1e21 < 1e-6
# ② 对称情形解析可知：N* = D* = sqrt(C/6)
N2, D2 = chinchilla_optimal(6e20, 400.0, 0.5, 400.0, 0.5)
assert abs(N2 - 1e10) / 1e10 < 1e-6 and abs(D2 - 1e10) / 1e10 < 1e-6
# ③ Chinchilla 算力下的已知例 + 最优性（扰动后 loss 不降）
C_chin = 5.76e23
Nc, Dc = chinchilla_optimal(C_chin, 406.4, 0.34, 410.7, 0.28)
assert 2e10 < Nc < 6e10, f"N*={Nc:.2e}"        # 公布常数给出 ~3.7e10（讲解 §4）
L_param = lambda N, D: 1.69 + 406.4 / N**0.34 + 410.7 / D**0.28
for k in [0.3, 0.5, 2.0, 3.0]:
    assert L_param(Nc, Dc) <= L_param(k * Nc, C_chin / (6 * k * Nc)) + 1e-12
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。三题分别对应 scaling 工作流的三块积木：记账 → 拟合 → 配比。

In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
def flops_per_token(N):
    return 6 * N            # 前向 2N + 反向 4N（dL/dx 和 dL/dW 各 2N）

def total_flops(N, D):
    return flops_per_token(N) * D

In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
def fit_power_law(Ns, losses):
    Ns = np.asarray(Ns, dtype=float)
    losses = np.asarray(losses, dtype=float)
    E0 = 0.9 * losses.min()
    slope, intercept = np.polyfit(np.log(Ns), np.log(losses - E0), 1)
    p0 = [float(np.exp(intercept)), float(-slope), float(E0)]
    def f(N, A, alpha, E):
        return A / N**alpha + E
    (A, alpha, E), _ = curve_fit(f, Ns, losses, p0=p0,
                                 bounds=([0, 0, 0], [np.inf, 5.0, losses.min()]),
                                 maxfev=20000)
    return float(A), float(alpha), float(E)

In [ ]:
# 参考答案 · 练习 3（先自己做，再对照）
def chinchilla_optimal(C, A, alpha, B, beta):
    G = (alpha * A / (beta * B)) ** (1.0 / (alpha + beta))
    N_opt = G * (C / 6.0) ** (beta / (alpha + beta))
    D_opt = (C / 6.0) ** (alpha / (alpha + beta)) / G
    return N_opt, D_opt

## 小结

你在 5 分钟 CPU 里走完了 scaling law 研究的闭环：

- **造点**：8 次真实训练（4 个尺寸 + 4 个数据量），每次都把 cosine schedule 对齐训练长度——Kaplan 与 Chinchilla 分歧的根源就是没做这一步 [Hoffmann 2022]；
- **拟合**：`curve_fit` 得到 $L(N)=A/N^\alpha+E$ 与 $L(D)=B/D^\beta+E$，并理解 $E$ 是语料的不可约熵（我们的语料连理论下界都能算出来）；
- **记账**：逐层数矩阵乘法验证 $C\approx6ND$，前向 2、反向 4；
- **配比**：解析解出 compute-optimal $N^*(C), D^*(C)$，并在 iso-FLOP 图上确认谷底落在解析线上；也看到了公布常数与 “$D/N\approx20$” 之间的方法张力。

记住三句话：**Kaplan 说加参数，Chinchilla 说各一半，推理经济学说小模型往死里训**（讲解 §5）；数据不够时重复 4 个 epoch 内几乎免费、16 个 epoch 后归零 [Muennighoff 2023]；loss 可预测不等于下游分数可预测（讲解 §7）。

**→ 模块 06 · KV Cache 与高效推理**：scaling law 告诉我们推理成本 $\approx 2N$/token 是大头，下一模块就动手把推理做快——KV cache、量化与 PagedAttention。

---
## 🎯 真实数据胶囊题：真实 Pythia 模型族的参数缩放定律

下载真实 Pythia 全家桶的 config，验证参数量 N 与隐藏维 h 的幂律关系（N ∝ h^α）。在 log-log 上做线性拟合，恢复指数 α —— 这是 scaling 的最基本结构。

> 本题为本模块新增的**真实数据**练习：自包含，直接用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, numpy as np
CACHE=os.path.expanduser("~/.llm_internals_data"); os.makedirs(CACHE,exist_ok=True)
def load_cfg(model, url):
    p=os.path.join(CACHE,f"{model}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"),I=c.get("intermediate_size",4*h))
PYTHIA={m:f"https://huggingface.co/EleutherAI/pythia-{m}/resolve/main/config.json"
        for m in ["160m","410m","1.4b","2.8b","6.9b","12b"]}

cfgs={m:load_cfg(f"pythia-{m}", PYTHIA[m]) for m in ["160m","410m","1.4b","2.8b","6.9b","12b"]}
def pcount(c):
    L,h,V,I=c["L"],c["h"],c["V"],c["I"]; return 2*V*h+L*(4*h*h+4*h+2*h*I+(I+h)+4*h)+h
hs=np.array([c["h"] for c in cfgs.values()], float)
Ns=np.array([pcount(c) for c in cfgs.values()], float)
print("真实 Pythia (h, N):", list(zip(hs.astype(int), (Ns/1e6).round().astype(int))))

**练习**：实现 `fit_powerlaw(x, y)`：在 log-log 上对 `y=a·x^α` 做最小二乘，返回 `(alpha, a)`。用它拟合 N vs h，恢复指数（理论上接近 2~3，因为每层 ∝ h²，但 embedding 是 ∝ h）。

In [ ]:
def fit_powerlaw(x, y):
    # TODO: 对 log x, log y 做一次多项式拟合(deg=1)；slope=alpha, exp(intercept)=a
    raise NotImplementedError


In [ ]:
# 自测
alpha, a = fit_powerlaw(hs, Ns)
assert 1.5 < alpha < 3.5, f"参数随 h 的缩放指数应在2附近(主导项∝h²)，得到 {alpha:.2f}"
# 拟合应能近似复现
pred = a*hs**alpha
assert np.all(np.abs(np.log(pred)-np.log(Ns)) < 0.5), "log空间拟合应贴合"
# 已知幂律应精确恢复
am,_ = fit_powerlaw(np.array([1.,2.,4.,8.]), np.array([1.,4.,16.,64.]))
assert abs(am-2.0) < 1e-6, "y=x^2 应恢复 alpha=2"
print(f"真实 Pythia 参数缩放指数 alpha = {alpha:.2f} ✓")


### 📖 参考答案

In [ ]:
def fit_powerlaw(x, y):
    slope, intercept = np.polyfit(np.log(x), np.log(y), 1)
    return slope, np.exp(intercept)
print("✓ scaling law 的第一步就是 log-log 上的直线拟合")